In [ ]:
"""Example notebook demonstrating WSOL methods."""

import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src.models import create_model
from src.utils import get_device, denormalize_image
from src.visualization import WSOLVisualizer

# Set device
device = get_device("auto")
print(f"Using device: {device}")

# Create models
models = {
    "Grad-CAM": create_model("gradcam", num_classes=10),
    "Grad-CAM++": create_model("gradcam++", num_classes=10),
    "Score-CAM": create_model("scorecam", num_classes=10),
    "CAM": create_model("cam", num_classes=10),
}

# Move models to device
for model in models.values():
    model.to(device)
    model.eval()

# Create synthetic image
image = torch.randn(1, 3, 224, 224).to(device)

# Generate attention maps for all models
attention_maps = {}
predictions = {}

for name, model in models.items():
    with torch.no_grad():
        logits = model(image)
        pred_class = torch.argmax(logits, dim=1)
        attention_map = model.get_attention_maps(image)
        
        attention_maps[name] = attention_map.cpu()
        predictions[name] = pred_class.cpu()

# Visualize results
visualizer = WSOLVisualizer("assets")

fig, axes = plt.subplots(2, len(models), figsize=(4 * len(models), 8))

for i, (name, attention_map) in enumerate(attention_maps.items()):
    # Original image
    img_denorm = denormalize_image(image.cpu())
    axes[0, i].imshow(img_denorm[0].permute(1, 2, 0))
    axes[0, i].set_title(f"Original Image")
    axes[0, i].axis("off")
    
    # Attention map
    axes[1, i].imshow(attention_map[0, 0], cmap="jet")
    axes[1, i].set_title(f"{name}\nPred: {predictions[name].item()}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.savefig("assets/comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("Comparison saved to assets/comparison.png")
